# Lab 10 *(Self-Study, Architecture Reference)*: Publishing Agents to Gemini Enterprise
Building Enterprise AI Agents on **Gemini Enterprise Agent Platform (GEAP)**: Pulse Telco Workshop

---

## 🎯 Objectives
By the end of Lab 06, **Pulse Concierge** was running as a managed service on **Agent Runtime** in the workshop region (*REGION*, default *europe-west4*), and the **NOC Agent** was running on Cloud Run speaking the A2A protocol. From an infrastructure perspective, both agents are production-ready. From an organizational perspective, however, employees still have no unified place to discover and talk to them, and security architects still need a mechanism to pass the signed-in employee's identity down to backend data sources.

**Gemini Enterprise** is Google Cloud's employee-facing workspace application. Publishing a backend agent registers it into the Gemini Enterprise assistant catalog so business users can invoke it from the same interface they use for enterprise search and document analysis.

This is an **architectural design and reading lab**. Because provisioning a Gemini Enterprise application requires an active commercial subscription and per-seat user licenses, the temporary workshop projects do not include a live Gemini Enterprise instance. Instead, we examine the end-to-end control plane and runtime data flow, and run one read-only code cell that generates the exact *agents-cli publish gemini* commands for your deployed workshop agents:
1. **End-to-end topology**: How the Gemini Enterprise employee application routes conversations to backend agents hosted on Agent Runtime (**ADK registration**) or on Cloud Run, GKE, and external clouds (**A2A registration**).
2. **Control-plane registration**: How *agents-cli publish gemini* registers agents against the Discovery Engine API and how the employee catalog differs from **Agent Registry**.
3. **Per-user identity propagation**: How **authorization resources** manage OAuth 2.0 user consent and refresh tokens, how that pattern applies to **Microsoft Entra ID and Microsoft 365**, and why server-side OAuth token management differs from an Entra On-Behalf-Of (OBO) token exchange.
4. **Architectural boundaries**: Practical constraints around bidirectional voice streaming, Microsoft Teams or email task ingestion, regional gateway placement, and dual licensing/consumption billing.

**Naming and API mapping:** In Google Cloud's October 2026 product taxonomy, **Gemini Enterprise** is the commercial employee application, while **Gemini platform** (formerly Vertex AI / GEAP) is the underlying runtime and governance platform. Because Gemini Enterprise was built on the Discovery Engine stack, its underlying REST endpoint is *discoveryengine.googleapis.com*, its IAM roles use the *roles/discoveryengine.** prefix, and the Agents CLI refers to the target resource via *publish gemini* and *--gemini-app-id*.

### Key references
* 📘 [Register and manage ADK agents hosted on Agent Runtime](https://docs.cloud.google.com/gemini/enterprise/docs/register-and-manage-an-adk-agent): primary reference for ADK registration and authorization resources
* 📡 [Register and manage A2A agents](https://docs.cloud.google.com/gemini/enterprise/docs/register-and-manage-an-a2a-agent)
* 🪪 [Gemini Enterprise: configure identity provider](https://docs.cloud.google.com/gemini/enterprise/docs/configure-identity-provider) and [Workforce Identity Federation with Microsoft Entra ID](https://docs.cloud.google.com/iam/docs/workforce-sign-in-microsoft-entra-id)
* 🏛️ [Agent Registry: manage agents](https://docs.cloud.google.com/agent-registry/manage-agents)
* 🛡️ [Route Gemini Enterprise traffic through Agent Gateway](https://docs.cloud.google.com/gemini-enterprise-agent-platform/govern/gateways/agent-gateway-ge-deploy)
* 💳 [Compare editions of Gemini Enterprise](https://docs.cloud.google.com/gemini/enterprise/docs/editions) and [Get subscriptions and assign licenses](https://docs.cloud.google.com/gemini/enterprise/docs/licenses)
* 📦 [Google Agents CLI on PyPI (google-agents-cli)](https://pypi.org/project/google-agents-cli/)

## 💶 Cost and Scaling

| Edition (official docs naming) | Monthly term | 1-year commitment term |
| :--- | :--- | :--- |
| Business | \$25 / user / month | \$252 / user / year (≈ \$21 per month) |
| Standard | \$35 / user / month | \$30 / user / month |
| Plus | \$60 / user / month | \$50 / user / month |
| Frontline | \$15 / user / month | \$12.50 / user / month |
| Pay-as-you-go | Data index \$5/GiB-month plus model usage at Gemini platform prices | No annual commitment |

**Billing mechanics and scaling considerations:**
* Per-seat list prices above are taken from the official Google Cloud SKU catalog (verified 2026-10-20; see [Compare editions of Gemini Enterprise](https://docs.cloud.google.com/gemini/enterprise/docs/editions) and [Get subscriptions and assign licenses](https://docs.cloud.google.com/gemini/enterprise/docs/licenses)). Note that the highest commercial seat tier in the product documentation is named **Plus**.
* **Model Armor is included** across all Gemini Enterprise seat editions, and a 30-day no-cost **Plus** trial is available ([Gemini Enterprise overview](https://cloud.google.com/gemini-enterprise)).
* The **Plus** and **Standard** tiers include **\$15** and **\$10** per user per month of AI developer tool credits, metered as a rolling weekly pool.
* Every request delegated from Gemini Enterprise into your published ADK or A2A agent additionally bills the backend agent's own Agent Runtime compute and Gemini model tokens in *REGION* and *eu*, exactly as calculated in Labs 02 to 06 ([Gemini platform pricing](https://cloud.google.com/products/gemini-enterprise-agent-platform/pricing)).

**Estimated cost for this lab: \$0.00** (architectural walkthrough and read-only CLI inspection; no resources are provisioned).

## 1. End-to-End Request Flow: From the Employee Front Door to Governed Data

When an employee opens the **Gemini Enterprise** web application, they authenticate with their corporate credentials before any agent is invoked. Google Cloud supports two enterprise identity architectures ([Configure identity provider](https://docs.cloud.google.com/gemini/enterprise/docs/configure-identity-provider)):
* **Google Identity (Cloud Identity or Google Workspace)**: User accounts exist in Google's directory, optionally federated with an external identity provider such as Microsoft Entra ID or Okta for OIDC or SAML 2.0 single sign-on.
* **Workforce Identity Federation**: External identity provider users (such as Microsoft Entra ID principals) authenticate directly into Google Cloud **without synchronizing user accounts** into Cloud Identity. Official Gemini Enterprise documentation requires Workforce Identity Federation with Entra ID when connecting Microsoft 365 grounding data sources such as SharePoint, OneDrive, and Outlook.

Once signed in, the employee selects **Pulse Concierge** or the **NOC Agent** from the Gemini Enterprise agent picker. The sequence diagram below traces how Gemini Enterprise mediates authentication, invokes the backend agent, and passes user-scoped OAuth tokens when configured:

```mermaid
sequenceDiagram
    autonumber
    actor Employee as 👤 Employee<br/>(Google identity or<br/>Workforce Identity Federation)
    participant GE as 🌟 Gemini Enterprise app
    participant Auth as 🔐 Authorization resource<br/>(OAuth 2.0 client at your IdP)
    participant Engine as 🎯 Pulse Concierge<br/>(ADK registration,<br/>Agent Runtime)
    participant A2A as 📡 NOC agent<br/>(A2A registration,<br/>Cloud Run)
    participant Data as 📊 BigQuery / other APIs

    Employee->>GE: Asks Pulse Concierge a question
    opt First use of an agent that has an authorization resource
        GE->>Employee: OAuth consent screen
        Employee->>Auth: Signs in and consents (authorization-code flow)
        Auth-->>GE: Access + refresh token for THIS employee
    end
    alt ADK registration (Agent Runtime)
        GE->>Engine: streamQuery on the reasoning engine (+ the employee's token, if configured)
        Engine->>Data: Tool call with the employee's token, or as the agent's own identity
        Engine-->>GE: Answer
    else A2A registration (Cloud Run, GKE, elsewhere)
        GE->>A2A: A2A message to the URL in the agent card
        A2A-->>GE: Task result
    end
    GE-->>Employee: Answer in the chat
```

## 2. Publishing Backend Agents with the Agents CLI

Once a Gemini Enterprise application exists in your project and the publisher holds the Discovery Engine Editor role (*roles/discoveryengine.editor*), registering a backend agent is a single idempotent CLI command. The Gemini Enterprise application's data residency location (*global*, *eu*, or *us*) is fixed when the application is created and forms part of its Discovery Engine resource path.

**Regional routing between EU Gemini Enterprise apps and regional Agent Runtime instances:** A Gemini Enterprise application created in the *eu* multi-region routes its Agent Runtime calls through Google's *europe-west1* control plane and can invoke Agent Runtime instances hosted in either *europe-west4* (our workshop region) or *europe-west1*. If you also attach an outbound **Agent Gateway** directly to an EU Gemini Enterprise application ([Route Gemini Enterprise traffic through Agent Gateway](https://docs.cloud.google.com/gemini-enterprise-agent-platform/govern/gateways/agent-gateway-ge-deploy)), that gateway resource must be provisioned in *europe-west1*.

**Pattern A: ADK registration for Pulse Concierge on Agent Runtime.** Run the command from inside the agent project directory. The CLI automatically reads the target Agent Runtime resource name from *deployment_metadata.json* (written by *agents-cli deploy* in Lab 02 and Lab 06):

```bash
cd pulse-concierge
agents-cli publish gemini --list     # lists the Gemini Enterprise apps in the project
agents-cli publish gemini \
  --gemini-app-id "projects/${PROJECT_NUMBER}/locations/<APP_LOCATION>/collections/default_collection/engines/<APP_ID>" \
  --registration-type adk \
  --display-name "Pulse Concierge" \
  --description "Pulse Telco analytics across CRM, billing and network data"
# optional, for per-user access (section 5): add --authorization-id "projects/${PROJECT_NUMBER}/locations/<APP_LOCATION>/authorizations/<AUTH_ID>"
```

Under the hood, the CLI sends a registration request to the Discovery Engine API (*.../engines/<APP_ID>/assistants/default_assistant/agents*) containing an *adkAgentDefinition* pointing to your reasoning engine, alongside *authorizationConfig.toolAuthorizations* whenever *--authorization-id* is supplied. Re-running the command updates the existing registration in place rather than creating duplicates.

**Pattern B: A2A registration for the NOC Agent on Cloud Run** (or any agent built in another framework or hosted outside Agent Runtime):

```bash
agents-cli publish gemini \
  --gemini-app-id "projects/${PROJECT_NUMBER}/locations/<APP_LOCATION>/collections/default_collection/engines/<APP_ID>" \
  --registration-type a2a \
  --agent-card-url "$PULSE_NETWORK_OPS_AGENT_CARD"
```

The CLI fetches the JSON agent card from the target URL and registers an *a2aAgentDefinition*. In Lab 06 we deployed the NOC service with *--no-invoker-iam-check* to simplify local testing. In production, enforce Cloud Run IAM authentication and grant the Cloud Run Invoker role (*roles/run.servicesInvoker*) to the project's Discovery Engine service agent (*service-<PROJECT_NUMBER>@gcp-sa-discoveryengine.iam.gserviceaccount.com*).

## 3. Control Plane Catalog vs Employee Front Door: Agent Registry vs Gemini Enterprise

Because both services maintain a list of agents, teams frequently ask why **Agent Registry** and **Gemini Enterprise** are separate registries:
* **Agent Registry** is the platform engineering control plane for infrastructure discovery and agent-to-agent wiring. Every agent deployed to Agent Runtime is registered in Agent Registry **automatically at deploy time** (*pulse-concierge* has been listed in *europe-west4* since Lab 02), and platform teams manually register custom Cloud Run MCP servers (Lab 03) and A2A services (Lab 06).
* **Gemini Enterprise** maintains the end-user assistant directory shown in the employee web and mobile application. Running *agents-cli publish gemini* exposes a selected backend agent to business users in a specific Gemini Enterprise app without modifying Agent Registry.

You can verify that *pulse-concierge* is already cataloged in Agent Registry even though you have never run *publish gemini*:

```bash
gcloud agent-registry agents list --project="$PROJECT_ID" --location="$REGION" --format="table(displayName,name.basename())"
```

## 4. Choosing Between ADK Registration and A2A Registration

| Dimension | ADK registration (*--registration-type adk*) | A2A registration (*--registration-type a2a*) |
| :--- | :--- | :--- |
| **Backend hosting environment** | Agent Runtime (*europe-west4* or *europe-west1*), running an ADK application | Cloud Run, GKE, another cloud provider, or on-premises; any agent framework |
| **Wire protocol from Gemini Enterprise** | Native invocation on the reasoning engine (*:streamQuery*) | Open A2A JSON-RPC messages sent to the URL declared in the agent card |
| **Workshop representative** | *pulse-concierge* | *pulse-network-ops-agent* |
| **Session and memory management** | Managed Agent Runtime Sessions and Memory Bank (Labs 04 to 06) | Implemented and managed by the remote A2A agent service |
| **Per-user authorization binding (*--authorization-id*)** | *authorizationConfig.toolAuthorizations* | *authorizationConfig.agentAuthorization* |
| **Network governance boundary** | Agent Gateway governs **outbound** traffic from the agent (Lab 09) | Cloud Run IAM or IAP in front of the service, plus the remote service's own egress controls |

**Agent Gateway placement rules for Gemini Enterprise:** When you attach an Agent Gateway directly to Gemini Enterprise, the gateway operates **strictly in egress mode** (governing outbound model and tool calls made by the application, not inbound client-to-app traffic), and its region must match the application's routing region (*europe-west1* for an EU multi-region Gemini Enterprise app; see [Route Gemini Enterprise traffic through Agent Gateway](https://docs.cloud.google.com/gemini-enterprise-agent-platform/govern/gateways/agent-gateway-ge-deploy)).

**Architectural decision rule:** Choose **ADK registration** when your team builds the agent in ADK and hosts it on Agent Runtime so you get native *streamQuery* integration, managed sessions, and tool-level OAuth authorizations. Choose **A2A registration** when federating an agent owned by another department, written in a non-ADK stack (LangGraph, CrewAI, Go, or Java), or hosted on Cloud Run or GKE like the NOC Agent from Lab 06.

## 5. Identity Propagation: Authorization Resources and Microsoft Entra ID

Across the workshop we enforced Pulse Telco's domain access policy: Alex (executive) is authorized to query CRM, billing, and network data, whereas Elena (customer care) is restricted to CRM data only. When Gemini Enterprise sits in front of your backend agent, **three distinct security principals** can appear in your architecture. Keeping their tokens and audit trails distinct is essential for security sign-off:

| Security principal | Token type on the wire | What Cloud Audit Logs record |
| :--- | :--- | :--- |
| **The agent's own identity**: Agent Identity *principal://agents.global.org-...* (Lab 02) | Workload access token minted automatically by Agent Runtime for the agent's principal | The Agent Identity appears as the caller (for example when querying BigQuery through the managed BigQuery MCP server in Labs 03 to 06) |
| **The human employee**, via a Gemini Enterprise **authorization resource** | The employee's delegated OAuth 2.0 access token obtained after interactive user consent | The individual employee appears as the principal calling the downstream API |
| **A workshop persona service account** (Lab 01 local simulation) | Short-lived access token minted by impersonating *persona-alex-exec* or *persona-elena-care* | The persona service account as caller plus the impersonating principal inside *serviceAccountDelegationInfo* (verified in Lab 01 Step 1.4e) |

### How a Gemini Enterprise Authorization Resource Works at Runtime
Per the [ADK agent registration documentation](https://docs.cloud.google.com/gemini/enterprise/docs/register-and-manage-an-adk-agent), propagating end-user OAuth credentials follows a three-step lifecycle:
1. **One-time OAuth client registration**: An administrator creates an OAuth 2.0 client at the identity provider and registers an authorization resource in Gemini Enterprise (*serverSideOauth2*, specifying the client ID, client secret, authorization URI, and token URI). The requested OAuth scopes are embedded in the authorization URI alongside *access_type=offline* and *prompt=consent* so Gemini Enterprise receives a refresh token.
2. **Binding at publish time**: When running *agents-cli publish gemini*, pass *--authorization-id projects/<NUMBER>/locations/<APP_LOCATION>/authorizations/<AUTH_ID>*. The CLI accepts one authorization ID per invocation, while the underlying REST field (*toolAuthorizations*) accepts an array if an agent requires tokens for multiple downstream systems.
3. **Runtime consent and token injection**: The first time an employee invokes the published agent in Gemini Enterprise, the UI presents an OAuth consent prompt. Once the employee approves, Gemini Enterprise stores the refresh token server-side, mints fresh access tokens on subsequent turns, and injects the employee's access token into the runtime context passed to your agent tool (consult current Gemini Enterprise documentation for the exact *tool_context* key name in your ADK version). Your tool code then authenticates to the downstream API **as the employee** rather than as the agent container:
   * **Google Cloud APIs (such as BigQuery)**: Wrap the injected access token in *google.oauth2.credentials.Credentials(token=...)* and pass it to the BigQuery client. This is identical to the Phase 2 client construction in Lab 01, except the token arrives from Gemini Enterprise's OAuth store instead of local service account impersonation.
   * **Microsoft Graph APIs**: Attach the injected token directly as an *Authorization: Bearer <token>* HTTP header on calls to *https://graph.microsoft.com/...*.

### Architectural Implications for Microsoft Entra ID and Microsoft 365
Because a Gemini Enterprise authorization resource is a standard OAuth 2.0 authorization-code client, you can point it at **Microsoft Entra ID** by registering an Entra application and configuring your tenant's endpoints (*https://login.microsoftonline.com/<tenant>/oauth2/v2.0/authorize* and *.../token*) with least-privilege Microsoft Graph scopes (such as *Mail.Read* or *Calendars.Read*). After the employee consents (or an Entra tenant administrator grants admin consent), your agent receives a Graph access token scoped strictly to **that employee's** Microsoft 365 permissions:
* **Security outcome**: The agent acts strictly within the signed-in user's Entra ID and M365 privileges, preventing cross-user data leakage.
* **Protocol distinction**: Mechanically, this flow is **delegated OAuth 2.0 authorization-code consent and server-side token refresh managed by Gemini Enterprise**, rather than an **Entra ID On-Behalf-Of (OBO) token exchange** (*urn:ietf:params:oauth:grant-type:jwt-bearer*). If a downstream architecture requires a true middle-tier OBO assertion exchange, you implement that exchange inside your agent or MCP server code.
* **Verification boundary**: Configuring non-Google OAuth endpoints (such as Entra ID) inside a Gemini Enterprise authorization resource is not tested in this workshop environment; validate token hand-off against your own Entra tenant during pilot setup.

In Lab 01's *Workshop analogue vs production* table, we mapped each workshop identity pattern to its production equivalent. Publishing to Gemini Enterprise completes that picture: corporate user authentication happens at the Gemini Enterprise front door, and user-scoped downstream API tokens are delivered through attached authorization resources.

## 6. Architectural Boundaries: What Gemini Enterprise Publishing Does Not Cover

Before positioning Gemini Enterprise as the single front door for every agent use case, account for three platform boundaries:
* **Real-time bidirectional voice (*run_live*)**: The Gemini Enterprise web and mobile application communicates with registered agents over request/response text streams and does not provide a bidirectional audio transport for Gemini Live agents (Labs 01 and 08). Publish your text-based analytical and workflow agents to Gemini Enterprise; expose live voice agents through a custom web, mobile, or SIP telephony front end that connects directly to your runner or Agent Runtime.
* **Inbound task triggers from Microsoft Teams or email**: Gemini Enterprise includes built-in Microsoft 365 data connectors (SharePoint, OneDrive, Outlook, Teams) that index M365 content *into* Gemini Enterprise for grounded search and Q&A. Those data connectors do not expose your custom ADK or A2A agent as a callable bot inside Microsoft Teams or as an email inbox listener. If business users want to @mention your agent inside a Teams channel or trigger workflows by forwarding an email, deploy a lightweight Teams bot or webhook relay (for example on Cloud Run or via n8n as shown in Lab 08) that calls your agent's *:streamQuery* or A2A endpoint.
* **Separate seat licensing and backend runtime billing**: Gemini Enterprise is licensed per user seat (or via pay-as-you-go data indexing), while every conversation routed from Gemini Enterprise to your custom ADK or A2A agent also consumes Agent Runtime compute and Gemini model tokens in your GCP project ([Compare editions](https://docs.cloud.google.com/gemini/enterprise/docs/editions), [Get subscriptions and assign licenses](https://docs.cloud.google.com/gemini/enterprise/docs/licenses)).

### 🧪 Try It Yourself (5 Minutes Each, No Gemini Enterprise App Needed)
1. **Inspect the CLI contract directly.** Run the code cell in Section 7 below and examine the *--help* output to verify three details: (a) which flag selects between ADK and A2A registration, (b) where the CLI finds the Agent Runtime resource ID when you omit the explicit engine flag, and (c) how many *--authorization-id* arguments a single CLI invocation accepts.
2. **Confirm control-plane vs front-door registration.** Run the *gcloud agent-registry agents list* command from Section 3 in a terminal. Notice that *pulse-concierge* is already present in Agent Registry even though *publish gemini* was never executed, because Agent Registry entries are created automatically when deploying to Agent Runtime.
3. **Design a least-privilege Entra ID authorization resource for Elena.** Sketch the configuration needed if Elena wants Pulse Concierge to summarize her customer follow-up emails in Exchange Online: specify the Entra app registration settings, the tenant *authorizationUri* and *tokenUri*, and the narrowest Microsoft Graph scope (*Mail.Read* rather than *Mail.ReadWrite*). What does Elena see on first use, and what can the agent access with her delegated token that it cannot access with its own Agent Identity?

## 7. Reference Publish Commands for Your Workshop Agents

The read-only code cell below loads your *workshop.env* coordinates, prints the *agents-cli publish gemini* commands populated with your project number and deployed agent endpoints, and displays the CLI help reference. Because the workshop project does not contain a Gemini Enterprise application, *<APP_LOCATION>* and *<APP_ID>* remain placeholders and no network write calls are made.

In [ ]:
# Step 10.1: Print the publish commands for your workshop agents (read-only) and the CLI help
import os, pathlib, subprocess
from dotenv import load_dotenv

REPO_ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "labs" else pathlib.Path.cwd()
load_dotenv(REPO_ROOT / "workshop.env", override=True)
os.environ["GOOGLE_API_USE_CLIENT_CERTIFICATE"] = "false"

PROJECT_NUMBER = os.environ["PROJECT_NUMBER"]
REGION         = os.environ.get("REGION", "europe-west4")
AGENT_NAME     = os.environ.get("AGENT_NAME", "pulse-concierge")
APP            = f"projects/{PROJECT_NUMBER}/locations/<APP_LOCATION>/collections/default_collection/engines/<APP_ID>"

print(f"# Workshop region: {REGION} (an 'eu' Gemini Enterprise app routes via europe-west1 and can invoke engines in {REGION} or europe-west1)")
print(f"# ADK registration of {AGENT_NAME}; the CLI reads the engine from {AGENT_NAME}/deployment_metadata.json:")
print(f"#   {os.environ.get('PULSE_AGENT_ENGINE', '(not deployed yet: run Lab 02)')}")
print(f"cd {AGENT_NAME}")
print(f'agents-cli publish gemini --gemini-app-id "{APP}" --registration-type adk \\')
print('  --display-name "Pulse Concierge" --description "Pulse Telco analytics across CRM, billing and network data"')
print(f'# per-user access: add --authorization-id "projects/{PROJECT_NUMBER}/locations/<APP_LOCATION>/authorizations/<AUTH_ID>"')
print("cd ..\n")
print("# A2A registration of the NOC agent on Cloud Run (Lab 06):")
print(f'agents-cli publish gemini --gemini-app-id "{APP}" --registration-type a2a \\')
print(f'  --agent-card-url "{os.environ.get("PULSE_NETWORK_OPS_AGENT_CARD", "(not deployed yet: run Lab 06)")}"\n')

try:
    print(subprocess.run(["agents-cli", "publish", "gemini", "--help"], capture_output=True, text=True).stdout)
except FileNotFoundError:
    print("agents-cli is not installed in this environment - install it as in Lab 00 (Install and verify the Agents CLI)")

## ✅ Lab 10 Complete: Architecture Summary

* **One CLI command, two integration protocols:** *agents-cli publish gemini* connects native ADK agents on Agent Runtime via *:streamQuery* (**ADK registration**) and connects remote agents on Cloud Run, GKE, or external stacks via their JSON agent card (**A2A registration**).
* **Two distinct registries:** **Agent Registry** is the platform engineering catalog populated automatically on Agent Runtime deployment, whereas the **Gemini Enterprise** assistant list is the curated employee-facing catalog populated by *publish gemini*.
* **Delegated per-user access via authorization resources:** Attaching an *authorization resource* prompts each employee for one-time OAuth 2.0 consent, stores refresh tokens in Gemini Enterprise, and passes user-scoped access tokens to your agent tools for downstream calls to Google Cloud APIs or Microsoft Graph. For Entra ID, this provides consent-based delegated OAuth access rather than an in-container OBO token exchange.
* **Channel and licensing boundaries:** Bidirectional live voice agents and inbound Microsoft Teams or email bots require custom front-end relays outside the Gemini Enterprise web app, and backend agent invocations consume standard Agent Runtime and model token quotas alongside Gemini Enterprise seat licenses.

👉 **Next:** **[Lab 11: Environment Teardown and Workshop Architecture Recap](lab11_cleanup.ipynb)**.